# TraceX LLM runs on Kaggle (Member 3)

Runs Qwen2.5-7B-Instruct and Qwen2.5-3B-Instruct (4-bit) for experiments E1-E3.
**Both models run on the same single GPU** so accuracy and speed are comparable.

**Before you start (once):** verify your phone number at https://www.kaggle.com/settings.

**Notebook settings (right panel -> Session options), every time:**
1. Accelerator: **GPU T4 x2**. The notebook pins everything to **one** T4 (GPU 0) so every run uses identical hardware.
2. Internet: **On**.

**Workflow**
1. Set `STAGE` and `MODEL` in the next cell. Always run `debug` (20 instances) before any full stage.
2. Short stages (`debug`): run interactively with *Run All*.
3. Long stages (`gate1`, `e1`, `e3`): use **Save Version -> Save & Run All (Commit)**. It runs in the background
   (up to 12 h) and keeps the output. Files from an interactive session are lost when the session stops.
4. Download `tracex_llm_results.zip` from the **Output** tab and give it to Member 3 to commit.

**Resuming after a timeout:** upload the last `tracex_llm_results.zip` as a Kaggle Dataset (any name), add it to
the notebook with *Add Input*, and run the same stage again. Finished instances are skipped.

In [ ]:
# ---- settings: change these two -------------------------------------------
STAGE = "debug"    # "debug" | "gate1" | "e1" | "e3"
MODEL = "qwen7b"   # "qwen7b" | "qwen3b"
# -----------------------------------------------------------------------------
REPO = "https://github.com/CipherEnigma/TraceX.git"
BRANCH = "Aayush"          # change to "main" once the LLM code is merged
REPO_DIR = "/tmp/TraceX"   # outside /kaggle/working so the output holds only the results zip
OUT_ZIP = "/kaggle/working/tracex_llm_results.zip"

## 1. GPU: pin to one T4

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # must be set before torch is imported
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

## 2. Code and dependencies

If the repo is **private**, add a read-only GitHub token as a Kaggle secret named `GITHUB_TOKEN`
(*Add-ons -> Secrets*). Never paste the token into a cell. A public repo needs no token.

In [ ]:
import subprocess
os.environ["GIT_TERMINAL_PROMPT"] = "0"   # fail fast instead of asking for a username

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass   # no secret attached: fine for a public repo
url = REPO.replace("https://", f"https://x-access-token:{token}@") if token else REPO

def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode:
        msg = (r.stderr or r.stdout).replace(token, "***") if token else (r.stderr or r.stdout)
        hint = ("" if token else "\nIf the repo is private, add a GITHUB_TOKEN Kaggle secret (see above). "
                "Also check Internet is On in Session options.")
        raise RuntimeError(f"git {args[0]} failed:\n{msg}{hint}")   # the token never appears here
    return r.stdout.strip()

if os.path.isdir(REPO_DIR):
    git("-C", REPO_DIR, "fetch", url, f"{BRANCH}:refs/remotes/origin/{BRANCH}")
    git("-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}")
else:
    git("clone", "--branch", BRANCH, url, REPO_DIR)
    git("-C", REPO_DIR, "remote", "set-url", "origin", REPO)   # do not keep the token in .git/config
COMMIT = git("-C", REPO_DIR, "rev-parse", "--short", "HEAD")
print("code at commit", COMMIT, "(token used)" if token else "(public clone)")

In [ ]:
!pip install -q -r {REPO_DIR}/requirements-llm.txt
import importlib.metadata as md
for p in ["torch", "transformers", "accelerate", "bitsandbytes", "pandas"]:
    print(p, md.version(p))

## 3. Data (Member 1's pipeline, about 2 minutes)
The pipeline is deterministic. The check below fails if the evaluation samples differ from the ones committed in the repo.

In [ ]:
os.chdir(REPO_DIR)
!python -m tracex_data all
diff = subprocess.run(["git", "status", "--porcelain", "--", "data/processed"], capture_output=True, text=True).stdout
assert not diff.strip(), f"processed samples differ from the repo:\n{diff}"
print("samples identical to the committed ones")

## 4. Restore earlier results (resume)

In [ ]:
import glob, shutil, zipfile, pathlib, sys
sys.path.insert(0, REPO_DIR)
from tracex_llm import config as C

restored = 0
for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
    with zipfile.ZipFile(z) as f:
        for n in f.namelist():
            dest = pathlib.Path(REPO_DIR) / n
            if n.startswith("results/llm/") and not n.endswith("/") and not dest.exists():
                dest.parent.mkdir(parents=True, exist_ok=True)
                dest.write_bytes(f.read(n)); restored += 1
for src in glob.glob("/kaggle/input/**/results/llm/**/*.*", recursive=True):  # unzipped uploads
    rel = src.split("results/llm/", 1)[1]
    dest = C.RESULTS_DIR / rel
    if not dest.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dest); restored += 1
print(f"restored {restored} file(s)")

## 5. Run the stage
| Stage | What | Instances |
|---|---|---|
| `debug` | NYC, both methods, 20 instances: real speed + sanity check | 2 x 20 |
| `gate1` | NYC, both methods, full 500 sample: reproduction check (Gate 1) | 2 x 500 |
| `e1` | NYC + TKY, both methods (E1, E2 come from the same records) | 4 x 500 |
| `e3` | Massive-STEPS Tokyo, both methods, names present / removed / aliased | 6 x 334 |

`e1` includes the `gate1` runs, which are skipped if already finished.

In [ ]:
STAGES = {
    "debug": [("nyc", m, "none", "sample_test_debug20") for m in C.METHODS],
    "gate1": [("nyc", m, "none", "sample_test_500") for m in C.METHODS],
    "e1":    [(d, m, "none", "sample_test_500") for d in ("nyc", "tky") for m in C.METHODS],
    "e3":    [("msteps_tokyo", m, c, "sample_test_500") for m in C.METHODS for c in C.NAME_CONDITIONS],
}

def save_zip():
    with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(C.RESULTS_DIR.rglob("*")):
            if f.is_file():
                z.write(f, "results/llm/" + f.relative_to(C.RESULTS_DIR).as_posix())
    print(f"[saved] {OUT_ZIP}")

from tracex_llm.run import Session, run_one
sess = Session(MODEL)
for dataset, method, condition, sample in STAGES[STAGE]:
    run_one(sess, dataset, method, condition, None, sample)
    save_zip()   # after every run, so a timeout never loses finished work

## 6. Speed: measured seconds per instance and projected GPU hours

In [ ]:
import json
FULL_PLAN = {  # instances per (model) for the whole project
    "e1": 2 * 2 * 500,   # NYC + TKY x 2 methods
    "e3": 2 * 3 * 334,   # 2 methods x 3 name conditions
}
rows = []
for meta in sorted(C.RESULTS_DIR.rglob("*.meta.json")):
    m = json.loads(meta.read_text(encoding="utf-8"))
    for s in m["sessions"]:
        if "sec_per_instance" in s:
            rows.append((m["run"], s["backend"].get("gpu"), s["todo"], s["sec_per_instance"]))
for r in rows:
    print(f"{r[0]:70s} {r[1]}  n={r[2]:4d}  {r[3]:.2f} s/inst")
mine = [(r[2], r[3]) for r in rows if f"__{MODEL}__" in r[0]]
if mine:
    spi = sum(n * s for n, s in mine) / sum(n for n, _ in mine)   # weighted by instances
    total = sum(FULL_PLAN.values())
    print(f"\n{MODEL}: mean {spi:.2f} s/instance -> full E1+E3 plan ({total} instances) "
          f"~ {spi * total / 3600:.1f} GPU-hours")

## 7. Gate 1: reproduction check (NYC, Qwen2.5-7B)
Brief: our 7B Accuracy@1 should land near the published 0.18-0.23 (Nishida et al., 5-30 demonstrations).
More than about 3 points below means a pipeline bug - fix before anything else.

In [ ]:
from tracex_llm.run import summarize
PUBLISHED = {"nyc": (0.18, 0.23), "tky": (0.16, 0.19)}
freq = json.loads((pathlib.Path(REPO_DIR) / "reports/retrieval/nyc_results.json").read_text())["prediction"]["frequency"]
print(f"frequency heuristic (Member 2), NYC sample_test_500: acc1={freq['acc1']:.3f}\n")
for sample in ("sample_test_debug20", "sample_test_500"):
    for method in C.METHODS:
        p = C.result_path("nyc", MODEL, method, sample=sample)
        if not p.exists():
            continue
        r = summarize(p.stem, p)
        lo, hi = PUBLISHED["nyc"]
        verdict = ("within" if lo <= r["acc1"] <= hi else
                   "ABOVE" if r["acc1"] > hi else
                   "below, but within 3 points" if r["acc1"] >= lo - 0.03 else "MORE THAN 3 POINTS BELOW")
        note = " (20 instances: too few to judge)" if sample.endswith("debug20") else ""
        print(f"    published {lo:.2f}-{hi:.2f}: {verdict}{note}\n")

## 8. Look at a few raw answers

In [ ]:
for p in sorted(C.RESULTS_DIR.rglob(f"*__{MODEL}__*.jsonl"))[:2]:
    print("==", p.stem)
    for line in p.read_text(encoding="utf-8").splitlines()[:3]:
        r = json.loads(line)
        print(f"  target={r['target']} ranked={r['parse']['ranked'][:5]} status={r['parse']['status']} "
              f"copied={r['parse']['copied_top1']}\n  raw: {r['raw'][:200]!r}")

## 9. Free the GPU (only if switching `MODEL` in the same session)

In [ ]:
import gc, torch
del sess
gc.collect(); torch.cuda.empty_cache()
print("GPU memory freed; change MODEL in the settings cell and re-run section 5")